## This is the code to train the model and acquire influence and loss for Sensitivity to Outliers Experiment

**Default Code**:   
The current default code is a runnable sample. It runs on the synthetic dataset generated with sklearn's make_classification function. The default version code provides the synthetic dataset with 16500 samples and 160 features in total. The separation is set to 5 to make sure the dataset is distinguishable by the model. All features are set to be informative to ensure they are of equal importance. The dataset has only two labels, so it is a binary classification problem. The default setting will then generate the training set and test set from the pool. The default training sample size is 16000, and the test size is 500. The number of features is set to 10. The model in default will be a Simple FeedForward Neural Network constructed by TensorFlow. The Influence Estimation methods we provide by default are the Influence Function and TracIn. If you simply press 'play', the default code will generate ranked influence lists for both Influence Function and TracIn with respect to the above mentioned setting in the root directory. The result lists could then be fed into other analyses.

**By default, this is almost the same code as the base code. Please refer to the base code for more detailed explanation.** The differences here are: 1. Train_sizes are limited to 5 choices. 2. The loss of the training points are stored as a file for later analysis.

**Guideline**:  
Read in / Construct Datasets -> **Choose the Training Sample Size** -> Pre-processing -> Model Training -> Influence Estimation -> Store the Ranked Influence lists -> Store the Loss Lists -> **Change the Training Sample Size and Repeat all the process** -> ... -> **After all the training and estimation, feed the results into the analysis code**  (Remember to change the file name in the last three block3 to save lists in different settings.)

# Import Area

Here is the area to place all the import codes. You don't need to change here unless you want to customise in later sections.

In [1]:
import tensorflow as tf
import keras
from keras.utils import to_categorical
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [8]:
from keras import Sequential
from keras.layers import Dense, BatchNormalization, Dropout
from keras.losses import CategoricalCrossentropy
from keras.optimizers import Adam

In [9]:
from deel.influenciae.common import InfluenceModel, ExactIHVP
from deel.influenciae.influence import FirstOrderInfluenceCalculator
from deel.influenciae.utils import ORDER
from deel.influenciae.trac_in import TracIn

In [10]:
import random
from keras.optimizers import SGD

In [11]:
from sklearn.metrics import pairwise_distances
from sklearn.manifold import MDS
import seaborn as sns
import matplotlib.pyplot as plt

In [12]:
from sklearn.datasets import make_classification

In [13]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [14]:
# tf.debugging.set_log_device_placement(False)

Train_Size: 1000, 2000, 4000, 8000, 16000  

# Dataset Construction Area

**You can use any dataset you wish here, either regression or classification. But in default, since we are using influenciae's IF and TC method, make sure they are split into train and test sets, and then stored as tensorflow dataset format. If you only want to change the dataset, you can only change the code in the first two blocks to read in/ generate your own dataset. But remember to have features X and target y before going to the third block. Also, if you wish to everything on your own, remember to add id inside the dataset.** Since our default code is for classification, the regression might need a lot of changes in all the following sections.


**Input**: Dataset chosen(Usually in Features X and Target y format)  
**Output**: Tensorflow format Train and Test Set  
**Guideline**: Input -> Turn into Dataframe and add ID -> Pre-Processing -> Change the format to Tensorflow -> Output

The default code now produces a synthetic dataset with 16500 pool, 160 features with binary classification problems. The later options will turn that into a 10 features, 16000 train set and 500 test set sample. Both sets will then be turned into TensorFlow format and will wait for training.

**If you just want to use our setting to test, the only thing you need to change here is the train size. In this experiment, all the other things are fixed, but the number of training sample is changing to multi-verify the experiement results.**

1. Set your default setting here. train_pool + test_size = Total Dataset Size. train_sizes determines the later subset data. Sep to make sure the dataset is distinguishable.

In [15]:
train_pool = 44722
test_size = 500
train_sizes=[44722]
seed=42
ratios = [(9,1), (8,2), (7,3), (6,4), (5,5)]

2. Construct the Synthetic Dataset with Make Classification here. **Could replace this with other datasets with X and y.**

In [16]:
from ucimlrepo import fetch_ucirepo 
adult = fetch_ucirepo(id=2) 
  
X = adult.data.features 
y = adult.data.targets 


In [17]:
df = pd.concat([X, y], axis=1)

In [18]:
df['income'] = df['income'].str.strip().str.replace('.', '', regex=False)
df = df.replace('?', np.nan)
print("Before dropna:", len(df))
df = df.dropna().reset_index(drop=True)
print("After dropna:", len(df))

Before dropna: 48842
After dropna: 45222


In [19]:
labels = df['income']
labels_encoded = (labels == '>50K').astype(int)  # 1 for '>50K', 0 for '<=50K'
df['label'] = labels_encoded
df = df.drop('income', axis=1)
df = df.drop('education',axis = 1)

In [20]:
work_trans = LabelEncoder()
df["workclass"] = work_trans.fit_transform(df["workclass"])

mar_trans = LabelEncoder()
df["marital-status"] = mar_trans.fit_transform(df["marital-status"])

occ_trans = LabelEncoder()
df["occupation"] = occ_trans.fit_transform(df["occupation"])

rel_trans = LabelEncoder()
df["relationship"] = rel_trans.fit_transform(df["relationship"])

race_trans = LabelEncoder()
df["race"] = race_trans.fit_transform(df["race"])

sex_trans = LabelEncoder()
df["sex"] = sex_trans.fit_transform(df["sex"])

nat_trans = LabelEncoder()
df["native-country"] = nat_trans.fit_transform(df["native-country"])

In [21]:
print(df)

       age  workclass  fnlwgt  education-num  marital-status  occupation  \
0       39          5   77516             13               4           0   
1       50          4   83311             13               2           3   
2       38          2  215646              9               0           5   
3       53          2  234721              7               2           5   
4       28          2  338409             13               2           9   
...    ...        ...     ...            ...             ...         ...   
45217   33          2  245211             13               4           9   
45218   39          2  215419             13               0           9   
45219   38          2  374983             13               2           9   
45220   44          2   83891             13               0           0   
45221   35          3  182148             13               2           3   

       relationship  race  sex  capital-gain  capital-loss  hours-per-week  \
0        

In [22]:
df['id'] = np.arange(1, len(df) + 1)
print(df)

       age  workclass  fnlwgt  education-num  marital-status  occupation  \
0       39          5   77516             13               4           0   
1       50          4   83311             13               2           3   
2       38          2  215646              9               0           5   
3       53          2  234721              7               2           5   
4       28          2  338409             13               2           9   
...    ...        ...     ...            ...             ...         ...   
45217   33          2  245211             13               4           9   
45218   39          2  215419             13               0           9   
45219   38          2  374983             13               2           9   
45220   44          2   83891             13               0           0   
45221   35          3  182148             13               2           3   

       relationship  race  sex  capital-gain  capital-loss  hours-per-week  \
0        

In [23]:
print(df)
print(df["label"].value_counts())

       age  workclass  fnlwgt  education-num  marital-status  occupation  \
0       39          5   77516             13               4           0   
1       50          4   83311             13               2           3   
2       38          2  215646              9               0           5   
3       53          2  234721              7               2           5   
4       28          2  338409             13               2           9   
...    ...        ...     ...            ...             ...         ...   
45217   33          2  245211             13               4           9   
45218   39          2  215419             13               0           9   
45219   38          2  374983             13               2           9   
45220   44          2   83891             13               0           0   
45221   35          3  182148             13               2           3   

       relationship  race  sex  capital-gain  capital-loss  hours-per-week  \
0        

3. Turn the X and y into dataframe for easy further processing. Add ID column to easy retrieve samples within IF/TC.

In [24]:
df_train_pool = df.iloc[:train_pool].reset_index(drop=True)
df_test = df.iloc[train_pool:].reset_index(drop=True)

In [25]:
print(df_train_pool.head())
print(df_test.head())

   age  workclass  fnlwgt  education-num  marital-status  occupation  \
0   39          5   77516             13               4           0   
1   50          4   83311             13               2           3   
2   38          2  215646              9               0           5   
3   53          2  234721              7               2           5   
4   28          2  338409             13               2           9   

   relationship  race  sex  capital-gain  capital-loss  hours-per-week  \
0             1     4    1          2174             0              40   
1             0     4    1             0             0              13   
2             1     4    1             0             0              40   
3             0     2    1             0             0              40   
4             5     2    0             0             0              40   

   native-country  label  id  
0              38      0   1  
1              38      0   2  
2              38      0   3 

4. Here, we choose the subset of the full dataset. By setting features_to_test, we have the subset feature size. By changing nested_train_dfs, we have different sample sizes.

In [26]:
nested_train_dfs = [df_train_pool.iloc[:size].reset_index(drop=True) for size in train_sizes]

Again, change here if you wish to change the number of train size

In [27]:
train_df = nested_train_dfs[0]

In [28]:
train_df["label"].value_counts()

0    33635
1    11087
Name: label, dtype: int64

In [29]:
X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]
IDs = X_train["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_train = X_train.drop(columns=["id"]).values.astype(np.float32)
X_train = np.hstack((X_train, IDs))
y_train = to_categorical(y_train.values,num_classes=2)

print(X_train.shape)

(44722, 14)


In [30]:
test_df = df_test
X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]
IDs = X_test["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_test = X_test.drop(columns=["id"]).values.astype(np.float32)
X_test = np.hstack((X_test, IDs))
y_test = to_categorical(y_test.values,num_classes=2)

print(X_test.shape)

(500, 14)


In [31]:
test_df["label"].value_counts()

0    379
1    121
Name: label, dtype: int64

5 (Optional) The following code below can display the samples distribution. Uncomment them to acquire the distribution plot.

In [32]:
# X_all = np.vstack([X_train, X_test])

# y_train_1d = np.argmax(y_train, axis=1)
# y_test_1d  = np.argmax(y_test, axis=1)
# y_all_1d = np.hstack([y_train_1d, y_test_1d])

# D = pairwise_distances(X_all) 

In [33]:
# X_mds = MDS(n_components=2, dissimilarity='precomputed', random_state=0).fit_transform(D)

In [34]:
# sns.scatterplot(x=X_mds[:,0], y=X_mds[:,1], hue=y_all_1d)
# plt.title("MDS – preserves original distances")

6. Now we have the train_ds and test_ds for training

In [35]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

2026-06-30 23:01:47.006090: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38477 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:31:00.0, compute capability: 8.0


# Training Area

**Could modify the model as you wish here. Again, in default, influenciae relies on TensorFlow, so use the TensorFlow model if you only want to change the model. Remember: Store the InfluenceModel into the model_list with the loss function. The InfluenceModel will be used to obtain influence later. If you don't change the estimation methods, then the final output at this step shall always be the model_list**

**Input**:Train and Test Set from Data Construction Section   
**Output**: Model List  
**Guideline**: Input -> Define the Model and Hyperparameters -> Train the Model -> Output

Always remember to train the model, get the influence model and store that in model list, unless you wish to change the estimation methods.

The default code now use the train and test set generated from the last section to train the model. The default hyperparameters are: 300 Epochs, Simple FeedForward Neural Network, CategoricalCrossEntropy Loss function, SGD optimizer. Within each epoch, the current model will be turned into an Influence Model and stored inside a model list. After the training, the model list will be passed to next section for influence estimation.

**The only difference here is the loss for the training samples are stored and this shall be saved in a file later**

In [36]:
from tensorflow.keras.regularizers import l2

1. **Could modify the model as you wish here as long as it is tensorflow.** Just remember: Store the InfluenceModel into the model_list with the loss function

In [37]:
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),  
    BatchNormalization(momentum=0.9),
    Dropout(0.0),
    Dense(8, activation='relu'),
    Dense(y_train.shape[1])
])
loss_fn = CategoricalCrossentropy(from_logits=True)
optimizer = SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss=loss_fn, optimizer=optimizer, metrics=['accuracy'])

epochs = 150
unreduced_loss_fn = CategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
model_list = []
model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))
for i in range(epochs):
  model.fit(train_ds.batch(256), epochs=1, validation_data=test_ds.batch(256), verbose=2)
  model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))
base_loss, acc = model.evaluate(test_ds.batch(32), verbose=2)
print(base_loss)

2026-06-30 23:01:48.647050: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:630] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


175/175 - 2s - loss: 0.5993 - accuracy: 0.7108 - val_loss: 0.5624 - val_accuracy: 0.7680 - 2s/epoch - 9ms/step
175/175 - 0s - loss: 0.5476 - accuracy: 0.7604 - val_loss: 0.5357 - val_accuracy: 0.7680 - 244ms/epoch - 1ms/step
175/175 - 0s - loss: 0.5377 - accuracy: 0.7671 - val_loss: 0.5308 - val_accuracy: 0.7760 - 241ms/epoch - 1ms/step
175/175 - 0s - loss: 0.5279 - accuracy: 0.7749 - val_loss: 0.5088 - val_accuracy: 0.7860 - 234ms/epoch - 1ms/step
175/175 - 0s - loss: 0.5197 - accuracy: 0.7827 - val_loss: 0.4903 - val_accuracy: 0.8080 - 234ms/epoch - 1ms/step
175/175 - 0s - loss: 0.5176 - accuracy: 0.7853 - val_loss: 0.5144 - val_accuracy: 0.7780 - 235ms/epoch - 1ms/step
175/175 - 0s - loss: 0.5142 - accuracy: 0.7882 - val_loss: 0.5238 - val_accuracy: 0.7840 - 239ms/epoch - 1ms/step
175/175 - 0s - loss: 0.5155 - accuracy: 0.7858 - val_loss: 0.4920 - val_accuracy: 0.8020 - 238ms/epoch - 1ms/step
175/175 - 0s - loss: 0.5131 - accuracy: 0.7912 - val_loss: 0.4876 - val_accuracy: 0.8040 - 

2. Store the loss for each training sample here. Important for this experiment

In [38]:
train_loss_values = unreduced_loss_fn(y_train, model.predict(X_train)).numpy()

1398/1398 [==============================] - 1s 617us/step


# Influence Estimation Area

**Again, you could use other influence analysis methods rather than IF/TC. You can also use any other Influence Function or TracIn implementation. Just Remember: 1. Make sure the package is unform throughout the framework. 2. Generate a Ranked influence list for each Influence Function and TracIn; Only the ranked influence list could be fed into the following analysis code.**

**The default code now use the model list, train set and test set to estimate the influence, and produce a ranked influence list for both IF and TC. The results are then saved in the root directory.**

**Input**:Model list from Training section, Train and Test Set from Data Construction Section   
**Output**: Two ranked Influence Lists for IF and TC. One Loss list. 
**Guideline**: Input -> Influence Estimation Methods -> Influence Matrix -> Output

In [42]:
from tqdm import tqdm

1. Influence Function: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use influence_matrix.

In [43]:
train_ids = []
test_ids = []
train_samples_np = np.array([x.numpy() for x, y in train_ds])
train_ids = [round(sample[-1] * 1e10) for sample in train_samples_np]

In [44]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

influence_model = model_list[-1]
ihvp_calculator = ExactIHVP(influence_model, train_ds.batch(64))
influence_calculator = FirstOrderInfluenceCalculator(influence_model, train_ds, ihvp_calculator)

influence_matrix = np.zeros((num_test_samples, num_train_samples))

samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(64), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in tqdm(enumerate(explanation_ds.as_numpy_iterator()),total=num_test_samples,desc="Computing influence"):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            influence_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(influence_matrix, axis=0).reshape(1, -1)
df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(df)

Computing influence: 100%|██████████| 500/500 [03:12<00:00,  2.60it/s]


       Train_ID     Score
0             1  0.076879
1             2  0.023612
2             3  0.077053
3             4  0.091548
4             5  0.104413
...         ...       ...
44717     44718  0.130200
44718     44719 -0.003942
44719     44720 -0.310106
44720     44721  0.012301
44721     44722  0.102407

[44722 rows x 2 columns]


2. TracIn: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use TracIn_matrix.

In [46]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

TracIn_matrix = np.zeros((num_test_samples, num_train_samples))
influence_calculator = TracIn(
    model_list, 0.001
)
samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(64), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in tqdm(enumerate(explanation_ds.as_numpy_iterator()),total=num_test_samples,desc="Computing influence"):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            TracIn_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(TracIn_matrix, axis=0).reshape(1, -1)
TracIn_df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(TracIn_df)

Computing influence: 100%|██████████| 500/500 [04:40<00:00,  1.78it/s]


       Train_ID     Score
0             1  0.023500
1             2  0.021150
2             3  0.023306
3             4  0.023817
4             5  0.025350
...         ...       ...
44717     44718  0.022225
44718     44719  0.020584
44719     44720 -0.081169
44720     44721  0.019936
44721     44722  0.023840

[44722 rows x 2 columns]


3. Here we turn both influence lists to the ranked influence lists and then store them for further processing. Also, we store the previous loss list here.

In [47]:
df_sorted = df.sort_values(by="Score", ascending=False).reset_index(drop=True)

TracIn_sorted = TracIn_df.sort_values(by="Score", ascending=False).reset_index(drop=True)

In [48]:
TracIn_sorted.to_csv("TC_Train_Set_1.csv",index = False)
df_sorted.to_csv("IF_Train_Set_1.csv",index = False)

In [49]:
df_train_loss = pd.DataFrame({
    'Train_ID': train_ids,
    'Loss': train_loss_values
}).sort_values('Loss', ascending=False).reset_index(drop=True)
print(df_train_loss)

       Train_ID       Loss
0         18699  13.415754
1          4187  12.457027
2         33503  12.081600
3          5919  11.317877
4         34226  11.031629
...         ...        ...
44717     11598   0.000000
44718     11592   0.000000
44719     23034   0.000000
44720     36965   0.000000
44721     30808   0.000000

[44722 rows x 2 columns]


In [50]:
df_train_loss.to_csv("Train_Loss_Train_Set_1.csv",index = False)